# Volume 4, Chapter 12: The arithmetic–geometric mean (Python companion)

Exercises first, solutions at the end. We watch the AGM converge, check Gauss's theorem, Landen's transformation and the lemniscate, measure ellipses and pendulums, test Legendre's relation, race three algorithms for $\pi$ with plain Python integers, and compute logarithms with the AGM. Requirements: mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P12.1 (★) Watching the AGM.** Print $|a_n-b_n|$ for the AGM of $(1,\sqrt2)$, $(1,0.01)$ and $(1,10^{-10})$ with 120-digit arithmetic. How many steps are "slow" before the quadratic phase starts, and how does that number depend on $b/a$ (Proposition 12.1)?

**P12.2 (★) Gauss, Landen, Gauss again.** Check $I(a,b)=\frac\pi{2M(a,b)}$ (Theorem 12.2) by quadrature. Iterate Landen's transformation (Corollary 12.3) on $K(0.99)$ and watch the modulus collapse. Compute $\varpi$ and $\Gamma(\frac14)$ with the AGM (Corollary 12.4, Exercise 12.4).

**P12.3 (★★) Ellipses and pendulums.** Implement the perimeter formula of Theorem 12.7 and compare it with `mpmath.ellipe`, with Euler's bounds (Exercise 12.2), and with Ramanujan's two approximations $\pi\bigl(3(a+b)-\sqrt{(3a+b)(a+3b)}\bigr)$ and $\pi(a+b)\bigl(1+\frac{3h}{10+\sqrt{4-3h}}\bigr)$, $h=\bigl(\frac{a-b}{a+b}\bigr)^2$, for $a=1$ and $b$ from $1$ down to $10^{-4}$. Then tabulate the period of a pendulum against its amplitude (Exercise 12.3).

**P12.4 (★) Legendre's relation.** Check Theorem 12.9 for several $k$, and the derivative formula for $K$ of Proposition 12.8 by numerical differentiation.

**P12.5 (★★) CS application: the race for $\pi$.** With plain Python integers (fixed point), implement three algorithms: Machin's formula $\pi=16\arctan\frac15-4\arctan\frac1{239}$, the Brent–Salamin algorithm (Theorem 12.10), and the Chudnovsky series summed by binary splitting. Check $20\,000$ digits against mpmath, then time all three for $N=20\,000$, $40\,000$, $80\,000$ digits. How does each time grow when $N$ doubles?

**P12.6 (★★) Logarithms by the AGM.** Use Exercise 12.5 with $x=2^{-m}$ to compute $\ln2$ from $\frac\pi{2M(1,2^{-m})}\approx(m+2)\ln2$. How many digits do you get as a function of $m$? Compare the error $\frac\pi{2M(1,x)}-\ln\frac4x$ with $x^2\ln\frac1x$.

## Solutions

### P12.1

In [ ]:
import math, sys, time
sys.set_int_max_str_digits(0)
import mpmath as mp
mp.mp.dps = 120

def agm_trace(a, b, steps=12):
    a, b = mp.mpf(a), mp.mpf(b)
    out = []
    for n in range(steps):
        out.append(a - b)
        a, b = (a + b) / 2, mp.sqrt(a * b)
    return a, out

for a0, b0 in ((1, mp.sqrt(2)), (1, mp.mpf('0.01')), (1, mp.mpf('1e-10'))):
    M, diffs = agm_trace(a0, b0)
    print(f"M({mp.nstr(a0, 3)}, {mp.nstr(b0, 6)}) = {mp.nstr(M, 25)}")
    print("   |a_n - b_n|:", " ".join(f"{float(abs(d)):.0e}" if d != 0 else "0" for d in diffs))

For $(1,\sqrt2)$ the quadratic phase starts at once: $0.4$, $0.02$, $3\cdot10^{-5}$, $10^{-10}$, $10^{-21}$, $10^{-43}$, $10^{-87}$, then the difference stalls at the working precision. For $(1,0.01)$ there are about three slow steps, and for $(1,10^{-10})$ about six: while $a_n\gg b_n$, the difference only halves, and $b_{n+1}/a_{n+1}\approx2\sqrt{b_n/a_n}$, so the ratio needs about $\log_2\log\frac ab$ steps to become of order $1$. After that, digits double.

### P12.2

In [ ]:
def I_quad(a, b):
    return mp.quad(lambda t: 1 / mp.sqrt(a**2 * mp.cos(t)**2 + b**2 * mp.sin(t)**2), [0, mp.pi / 4, mp.pi / 2])

mp.mp.dps = 30
for a, b in ((1, 2), (3, 0.5), (1, 0.001)):
    a, b = mp.mpf(a), mp.mpf(b)
    print(f"I({a},{b}) = {mp.nstr(I_quad(a, b), 20)}   pi/(2M) = {mp.nstr(mp.pi / (2 * mp.agm(a, b)), 20)}")

# Landen: K(k) = (1 + k1) K(k1), iterated
k = mp.mpf('0.99')
prod, kk = mp.mpf(1), k
for step in range(6):
    kp = mp.sqrt(1 - kk**2)
    k1 = (1 - kp) / (1 + kp)
    prod *= 1 + k1
    kk = k1
    print(f"  Landen step {step + 1}: modulus {mp.nstr(kk, 5)}   product * pi/2 = {mp.nstr(prod * mp.pi / 2, 20)}")
print("  K(0.99) =", mp.nstr(mp.ellipk(k**2), 20))

varpi = mp.pi / mp.agm(1, mp.sqrt(2))
print("varpi =", mp.nstr(varpi, 20), "  2*int_0^1 dr/sqrt(1-r^4) =", mp.nstr(2 * mp.quad(lambda r: 1 / mp.sqrt(1 - r**4), [0, 1]), 20))
print("Gamma(1/4) =", mp.nstr((2 * mp.pi)**0.75 / mp.sqrt(mp.agm(1, mp.sqrt(2))), 20), "  mpmath:", mp.nstr(mp.gamma(0.25), 20))

Quadrature and the AGM agree to all twenty digits, even for $b=0.001$, where the integrand has a sharp peak. Landen's transformation squares the modulus at each step ($0.99$, $0.75$, $0.21$, $0.011$, $3\cdot10^{-5}$, $2\cdot10^{-10}$), and the product of the factors $1+k_n$, times $\frac\pi2$, converges quadratically to $K(0.99)$: it is the AGM in disguise. The two values of $\varpi$ differ in the $17$th digit, an error of the quadrature, which struggles with the singularity at $r=1$; the AGM value is the correct one, and $\Gamma(\frac14)$ comes out to all twenty digits.

### P12.3

In [ ]:
def perimeter(a, b):
    """Theorem 12.7: L = 2 pi (a^2 - sum 2^(n-1) c_n^2) / M(a, b), for a >= b > 0."""
    a, b = mp.mpf(a), mp.mpf(b)
    s = (a * a - b * b) / 2
    A, B, n = a, b, 0
    while abs(A - B) > mp.mpf(10)**(-mp.mp.dps + 3):
        c = (A - B) / 2
        A, B = (A + B) / 2, mp.sqrt(A * B)
        n += 1
        s += 2**(n - 1) * c * c
    return 2 * mp.pi * (a * a - s) / A

def ramanujan1(a, b):
    return mp.pi * (3 * (a + b) - mp.sqrt((3 * a + b) * (a + 3 * b)))

def ramanujan2(a, b):
    h = ((a - b) / mp.mpf(a + b))**2
    return mp.pi * (a + b) * (1 + 3 * h / (10 + mp.sqrt(4 - 3 * h)))

print("  b      L(1,b)              4E check    Euler low   Euler high   Ramanujan I   Ramanujan II")
for b in ('1', '0.9', '0.5', '0.1', '0.01', '0.0001'):
    b = mp.mpf(b)
    L = perimeter(1, b)
    chk = 4 * mp.ellipe(1 - b**2)
    print(f"{mp.nstr(b, 4):>7}  {mp.nstr(L, 15):>18}  {float(abs(L - chk)):.0e}   "
          f"{float((mp.pi * (1 + b) - L) / L):+.1e}   {float((mp.pi * mp.sqrt(2 * (1 + b * b)) - L) / L):+.1e}   "
          f"{float((ramanujan1(1, b) - L) / L):+.1e}     {float((ramanujan2(1, b) - L) / L):+.1e}")
print()
print("pendulum: T / (2 pi sqrt(l/g)) = 1 / M(1, cos(theta0/2))")
for deg in (5, 10, 30, 60, 90, 120, 150, 179):
    t = mp.radians(deg)
    print(f"  {deg:3d} degrees: {mp.nstr(1 / mp.agm(1, mp.cos(t / 2)), 8)}")

The AGM formula matches `ellipe` to $30$ digits. Euler's bounds are within $0.07\%$ for $b=0.9$ but $20\%$ off for flat ellipses. Ramanujan's first approximation errs by $3\cdot10^{-6}$ at $b=0.5$ and by at most $0.42\%$ (for the flat segment, perimeter $4$); his second by $5\cdot10^{-10}$ at $b=0.5$ and $0.04\%$ at worst. Both always underestimate here. For the pendulum, the small-angle period is $0.05\%$ too short at $5^\circ$, $18\%$ at $90^\circ$, and the period diverges as the amplitude approaches $180^\circ$, where the pendulum balances upside down.

### P12.4

In [ ]:
for k in ('0.1', '0.5', mp.sqrt(mp.mpf(1) / 2), '0.9', '0.999'):
    k = mp.mpf(k)
    kp = mp.sqrt(1 - k**2)
    K, E, Kp, Ep = mp.ellipk(k**2), mp.ellipe(k**2), mp.ellipk(kp**2), mp.ellipe(kp**2)
    print(f"k = {mp.nstr(k, 6):>8}:  EK' + E'K - KK' - pi/2 = {float(E * Kp + Ep * K - K * Kp - mp.pi / 2):+.1e}")
k = mp.mpf('0.6'); kp2 = 1 - k**2
dK = mp.diff(lambda x: mp.ellipk(x**2), k)
print("dK/dk numerically:", mp.nstr(dK, 15), "  (E - k'^2 K)/(k k'^2):", mp.nstr((mp.ellipe(k**2) - kp2 * mp.ellipk(k**2)) / (k * kp2), 15))

Legendre's relation holds to the working precision everywhere, including $k=0.999$, where $K$ is large and $K'$ close to $\frac\pi2$. The derivative formula agrees with numerical differentiation to $15$ digits.

### P12.5

In [ ]:
def pi_machin(digits):
    """pi = 16 arctan(1/5) - 4 arctan(1/239), in fixed point with Python integers."""
    unity = 10**(digits + 10)
    def arctan_inv(x):
        total, term, x2, n, sign = 0, unity // x, x * x, 1, 1
        while term:
            total += sign * (term // n)
            term //= x2
            n += 2
            sign = -sign
        return total
    return (16 * arctan_inv(5) - 4 * arctan_inv(239)) // 10**10

def pi_brent_salamin(digits):
    """Theorem 12.10 in fixed point: a, b, c are integers scaled by 10^(digits+10)."""
    one = 10**(digits + 10)
    a, b = one, math.isqrt(one * one // 2)
    s, n = 0, 0
    while abs(a - b) > 1:
        c = (a - b) // 2
        a, b = (a + b) // 2, math.isqrt(a * b)
        n += 1
        s += 2**(n + 1) * c * c // one
    return (4 * a * a // (one - s)) // 10**10, n

def pi_chudnovsky(digits):
    """Chudnovsky series, summed by binary splitting (exact integers until the last division)."""
    C3_24 = 640320**3 // 24
    def bs(a, b):
        if b - a == 1:
            if a == 0:
                P = Q = 1
            else:
                P = (6 * a - 5) * (2 * a - 1) * (6 * a - 1)
                Q = a * a * a * C3_24
            T = P * (13591409 + 545140134 * a)
            if a % 2:
                T = -T
            return P, Q, T
        m = (a + b) // 2
        P1, Q1, T1 = bs(a, m)
        P2, Q2, T2 = bs(m, b)
        return P1 * P2, Q1 * Q2, T1 * Q2 + P1 * T2
    terms = digits // 14 + 2
    P, Q, T = bs(0, terms)
    one = 10**(digits + 10)
    sqrtC = math.isqrt(10005 * one * one)
    return (Q * 426880 * sqrtC // T) // 10**10

def check(x, digits, ref):
    s = str(x)
    good = 0
    while good < len(s) and good < len(ref) and s[good] == ref[good]:
        good += 1
    return good - 1   # digits after the leading 3

N = 20000
mp.mp.dps = N + 50
ref = mp.nstr(mp.pi, N + 20, strip_zeros=False).replace(".", "")
for name, f in (("Machin", pi_machin), ("Brent-Salamin", lambda d: pi_brent_salamin(d)[0]), ("Chudnovsky", pi_chudnovsky)):
    t0 = time.perf_counter(); x = f(N); t = time.perf_counter() - t0
    print(f"{name:14s} {N} digits: {t:7.3f} s, correct digits {check(x, N, ref)}")
print("Brent-Salamin AGM steps for", N, "digits:", pi_brent_salamin(N)[1])
mp.mp.dps = 30

In [ ]:
for N in (20000, 40000, 80000):
    times = []
    for f in (pi_machin, lambda d: pi_brent_salamin(d)[0], pi_chudnovsky):
        t0 = time.perf_counter(); f(N); times.append(time.perf_counter() - t0)
    print(f"N = {N:6d}:  Machin {times[0]:7.2f} s   Brent-Salamin {times[1]:6.2f} s   Chudnovsky {times[2]:6.3f} s")

All three algorithms give $20\,000$ correct digits, and Brent–Salamin needs only $14$ AGM steps. When $N$ doubles, Machin's time is multiplied by about $4$: it is $O(N^2)$, with about $N$ terms each costing $O(N)$. Brent–Salamin and Chudnovsky grow by a factor of about $3$, because their cost is dominated by multiplications of $N$-digit integers, and Python multiplies big integers with Karatsuba's $O(N^{1.58})$ algorithm (Exercise 7.3). With FFT multiplication (Chapter 7), as in GMP or y-cruncher, the factor would approach $2$. Chudnovsky wins by a constant factor: each term gives $14$ digits, and binary splitting keeps all intermediate numbers exact and balanced.

### P12.6

In [ ]:
mp.mp.dps = 120
L2 = mp.log(2)
print(" m    correct digits of ln 2 from (m+2) ln 2 = pi / (2 M(1, 2^-m))")
for m in (10, 20, 50, 100, 150):
    approx = mp.pi / (2 * mp.agm(1, mp.mpf(2)**(-m))) / (m + 2)
    err = abs(approx - L2)
    print(f"{m:3d}   {float(-mp.log10(err)):6.1f}     (0.6 m = {0.6 * m:5.1f})")
for x in ('0.1', '0.01', '1e-6'):
    x = mp.mpf(x)
    e = mp.pi / (2 * mp.agm(1, x)) - mp.log(4 / x)
    print(f"x = {mp.nstr(x, 2):>6}: pi/(2M(1,x)) - ln(4/x) = {float(e):.2e}   x^2 ln(1/x) = {float(x * x * mp.log(1 / x)):.2e}")

Each $m$ gives about $0.6m$ digits of $\ln2$, since the error is of order $x^2\ln\frac1x$ with $x=2^{-m}$, that is $4^{-m}$ up to a factor $m$; the table shows the error is a few times below $x^2\ln\frac1x$. To get $N$ digits one takes $m\approx1.7N$, and the AGM of $1$ and $2^{-m}$ needs about $\log_2m$ slow steps and $\log_2N$ fast ones. This is Brent's method for logarithms, and through Newton's method for the exponential and the complex AGM, for all elementary functions at the speed of multiplication.